# Stock Price Prediction using LSTM, CNN, and Transformer

Predicting stock prices for 17 commercial banks using deep learning models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Conv1D, MaxPooling1D, Flatten, Bidirectional,
    Input, LayerNormalization, GlobalAveragePooling1D, MultiHeadAttention, Add
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")

## Load Data

In [ ]:
data = pd.read_csv("../data_preprocessing/combined_banks_dataset.csv")
data['published_date'] = pd.to_datetime(data['published_date'])
data = data.sort_values(['company_id', 'published_date']).reset_index(drop=True)

print(f"Shape: {data.shape}")
print(f"Banks: {data['company_id'].nunique()}")
print(f"Date range: {data['published_date'].min()} to {data['published_date'].max()}")

## Create Sequences

In [ ]:
def create_sequences_per_company(data, seq_length):
    X, y, company_ids, dates = [], [], [], []
    
    for company in data['company_id'].unique():
        company_data = data[data['company_id'] == company].sort_values('published_date')
        prices = company_data['close'].values
        dates_arr = company_data['published_date'].values
        
        for i in range(seq_length, len(prices)):
            X.append(prices[i-seq_length:i])
            y.append(prices[i])
            company_ids.append(company)
            dates.append(dates_arr[i])
    
    return np.array(X), np.array(y), np.array(company_ids), np.array(dates)

SEQ_LENGTH = 60

X, y, company_ids, dates = create_sequences_per_company(data, SEQ_LENGTH)
print(f"X shape: {X.shape}, y shape: {y.shape}")

## Train-Test Split

In [ ]:
min_date = dates.min()
max_date = dates.max()
cutoff_date = min_date + (max_date - min_date) * 0.8

train_mask = dates < cutoff_date
test_mask = dates >= cutoff_date

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

## Scale Data

In [ ]:
scaler = MinMaxScaler()
scaler.fit(y_train.reshape(-1, 1))

def scale_sequences(X, scaler):
    X_scaled = np.zeros_like(X, dtype=np.float32)
    for i in range(len(X)):
        X_scaled[i] = scaler.transform(X[i].reshape(-1, 1)).flatten()
    return X_scaled

X_train_scaled = scale_sequences(X_train, scaler)
y_train_scaled = scaler.transform(y_train.reshape(-1, 1)).flatten()

X_test_scaled = scale_sequences(X_test, scaler)
y_test_scaled = scaler.transform(y_test.reshape(-1, 1)).flatten()

X_train_3d = X_train_scaled.reshape(-1, SEQ_LENGTH, 1)
X_test_3d = X_test_scaled.reshape(-1, SEQ_LENGTH, 1)

print(f"Train shape: {X_train_3d.shape}")
print(f"Test shape: {X_test_3d.shape}")

## Model Definitions

In [ ]:
def build_lstm_model(input_shape):
    model = Sequential([
        Bidirectional(LSTM(100, return_sequences=True), input_shape=input_shape),
        Dropout(0.2),
        Bidirectional(LSTM(100, return_sequences=True)),
        Dropout(0.2),
        Bidirectional(LSTM(50)),
        Dropout(0.2),
        Dense(50, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='huber', metrics=['mae'])
    return model

In [ ]:
def build_cnn_model(input_shape):
    model = Sequential([
        Conv1D(128, 3, activation='relu', padding='same', input_shape=input_shape),
        Conv1D(128, 3, activation='relu', padding='same'),
        MaxPooling1D(2),
        Dropout(0.2),
        
        Conv1D(64, 3, activation='relu', padding='same'),
        Conv1D(64, 3, activation='relu', padding='same'),
        MaxPooling1D(2),
        Dropout(0.2),
        
        Conv1D(32, 3, activation='relu', padding='same'),
        GlobalAveragePooling1D(),
        
        Dense(100, activation='relu'),
        Dropout(0.3),
        Dense(50, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='huber', metrics=['mae'])
    return model

In [ ]:
def transformer_block(inputs, head_size, num_heads, ff_dim, dropout=0.2):
    x = LayerNormalization(epsilon=1e-6)(inputs)
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = Dropout(dropout)(x)
    res = Add()([x, inputs])
    
    x = LayerNormalization(epsilon=1e-6)(res)
    x = Dense(ff_dim, activation='relu')(x)
    x = Dropout(dropout)(x)
    x = Dense(inputs.shape[-1])(x)
    x = Dropout(dropout)(x)
    return Add()([x, res])

def build_transformer_model(input_shape):
    inputs = Input(shape=input_shape)
    
    # Positional encoding
    x = Dense(64)(inputs)
    
    # Transformer blocks
    x = transformer_block(x, head_size=64, num_heads=4, ff_dim=128, dropout=0.2)
    x = transformer_block(x, head_size=64, num_heads=4, ff_dim=128, dropout=0.2)
    x = transformer_block(x, head_size=64, num_heads=4, ff_dim=128, dropout=0.2)
    
    x = GlobalAveragePooling1D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1)(x)
    
    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='huber', metrics=['mae'])
    return model

## Training Setup

In [ ]:
EPOCHS = 100
BATCH_SIZE = 32

callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
]

## Train LSTM

In [ ]:
print("Training LSTM...")
lstm_model = build_lstm_model((SEQ_LENGTH, 1))
history_lstm = lstm_model.fit(
    X_train_3d, y_train_scaled,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

## Train CNN

In [ ]:
print("Training CNN...")
cnn_model = build_cnn_model((SEQ_LENGTH, 1))
history_cnn = cnn_model.fit(
    X_train_3d, y_train_scaled,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

## Train Transformer

In [ ]:
print("Training Transformer...")
transformer_model = build_transformer_model((SEQ_LENGTH, 1))
history_transformer = transformer_model.fit(
    X_train_3d, y_train_scaled,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

## Evaluation

In [ ]:
def evaluate_model(model, name, X_test, y_test_scaled, y_test_actual, scaler):
    pred_scaled = model.predict(X_test, verbose=0).flatten()
    pred = scaler.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
    
    rmse = np.sqrt(mean_squared_error(y_test_actual, pred))
    mae = mean_absolute_error(y_test_actual, pred)
    r2 = r2_score(y_test_actual, pred)
    mape = np.mean(np.abs((y_test_actual - pred) / y_test_actual)) * 100
    
    print(f"\n{name}:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  R2:   {r2:.4f}")
    print(f"  MAPE: {mape:.2f}%")
    
    return pred, rmse, mae, r2, mape

print("="*70)
print("EVALUATION RESULTS")
print("="*70)

pred_lstm, rmse_lstm, mae_lstm, r2_lstm, mape_lstm = evaluate_model(
    lstm_model, "LSTM", X_test_3d, y_test_scaled, y_test, scaler
)

pred_cnn, rmse_cnn, mae_cnn, r2_cnn, mape_cnn = evaluate_model(
    cnn_model, "CNN", X_test_3d, y_test_scaled, y_test, scaler
)

pred_trans, rmse_trans, mae_trans, r2_trans, mape_trans = evaluate_model(
    transformer_model, "Transformer", X_test_3d, y_test_scaled, y_test, scaler
)

## Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['LSTM', 'CNN', 'Transformer'],
    'RMSE': [rmse_lstm, rmse_cnn, rmse_trans],
    'MAE': [mae_lstm, mae_cnn, mae_trans],
    'R2': [r2_lstm, r2_cnn, r2_trans],
    'MAPE': [mape_lstm, mape_cnn, mape_trans]
}).sort_values('RMSE')

print("\n" + "="*70)
print("MODEL RANKING")
print("="*70)
print(results.to_string(index=False))
print(f"\nBest: {results.iloc[0]['Model']}")

## Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = [
    ('LSTM', pred_lstm),
    ('CNN', pred_cnn),
    ('Transformer', pred_trans)
]

for idx, (name, pred) in enumerate(models):
    axes[idx].plot(y_test[:200], label='Actual', linewidth=2)
    axes[idx].plot(pred[:200], label='Predicted', linewidth=2, alpha=0.7)
    axes[idx].set_title(f'{name}')
    axes[idx].set_xlabel('Sample')
    axes[idx].set_ylabel('Price')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].bar(results['Model'], results['RMSE'])
axes[0,0].set_title('RMSE')
axes[0,0].grid(True, alpha=0.3, axis='y')

axes[0,1].bar(results['Model'], results['MAE'])
axes[0,1].set_title('MAE')
axes[0,1].grid(True, alpha=0.3, axis='y')

axes[1,0].bar(results['Model'], results['R2'])
axes[1,0].set_title('R2')
axes[1,0].grid(True, alpha=0.3, axis='y')

axes[1,1].bar(results['Model'], results['MAPE'])
axes[1,1].set_title('MAPE %')
axes[1,1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

histories = [
    ('LSTM', history_lstm),
    ('CNN', history_cnn),
    ('Transformer', history_transformer)
]

for idx, (name, hist) in enumerate(histories):
    axes[idx].plot(hist.history['loss'], label='Train')
    axes[idx].plot(hist.history['val_loss'], label='Val')
    axes[idx].set_title(f'{name} Training')
    axes[idx].set_xlabel('Epoch')
    axes[idx].set_ylabel('Loss')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()